## Task 4.1: Experiment Tracking with MLflow

In this notebook, we track machine learning experiments using MLflow.  
According to the coursework, MLflow must be used to record:

- model parameters
- evaluation metrics
- preprocessing pipeline and model artifacts
- environment/dependency information
- multiple experiment runs for comparison

This notebook tracks experiments for:
1. Price prediction (regression)
2. Delivery status classification
3. Customer segment classification

The final goal is to compare runs and identify the best-performing customer segment model for deployment.

In [42]:
print("Task 4.1 - Experiment Tracking with MLflow")

Task 4.1 - Experiment Tracking with MLflow


In [43]:
import os
import joblib
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.linear_model import SGDRegressor, LogisticRegression

import mlflow
import mlflow.sklearn

In [44]:
# Create folders for artifacts and mlruns
os.makedirs("../artifacts", exist_ok=True)
os.makedirs("../mlruns", exist_ok=True)

In [45]:
# Set MLflow tracking URI and experiment
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("AuraCart_ML_Experiments")

print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: file:../mlruns


In [46]:
# Load dataset
df = pd.read_csv("../data/cleaned_data.csv")
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print("Dataset shape:", df.shape)
print(df.columns.tolist())
df.head()

Dataset shape: (10000, 17)
['category', 'price', 'quantity', 'delivery_status', 'payment_method', 'device_type', 'channel', 'customer_segment', 'order_year', 'order_month', 'order_day', 'order_dayofweek', 'order_hour', 'shipping_year', 'shipping_month', 'shipping_day', 'shipping_time']


,category,price,quantity,delivery_status,payment_method,device_type,channel,customer_segment,order_year,order_month,order_day,order_dayofweek,order_hour,shipping_year,shipping_month,shipping_day,shipping_time
0,Books,45.95,4,Shipped,PayPal,Mobile,Paid Search,VIP,2024,4,20,5,14,2024,4,27,7
1,Electronics,403.17,3,Delivered,PayPal,Mobile,Paid Search,Returning,2024,4,20,5,14,2024,4,22,2
2,Beauty,317.45,2,Shipped,Credit Card,Mobile,Email,Returning,2024,4,20,5,14,2024,4,27,7
3,Home,24.08,3,Shipped,PayPal,Tablet,Social,VIP,2024,4,20,5,14,2024,4,24,4
4,Clothing,494.90,1,Delivered,PayPal,Tablet,Organic,VIP,2024,4,20,5,14,2024,4,25,5


## Common preprocessing setup

To keep experiment tracking reproducible, we use Scikit-learn Pipelines and ColumnTransformer.  
This ensures the exact same preprocessing is applied during training and later deployment. :contentReference[oaicite:1]{index=1}

In [47]:
# Reusable preprocessor function
def build_preprocessor(X):
    categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    numerical_cols = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ])

    return preprocessor, categorical_cols, numerical_cols

                Part A — Regression Experiments

In [48]:
# Prepare regression data
reg_df = df.copy()

# safer setup: remove classification targets from regression inputs
y_reg = reg_df["price"]
X_reg = reg_df.drop(columns=["price", "delivery_status", "customer_segment"], errors="ignore")

print("Regression X shape:", X_reg.shape)
print("Regression y shape:", y_reg.shape)

Regression X shape: (10000, 14)
Regression y shape: (10000,)


In [49]:
# Split regression data
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

preprocessor_reg, reg_cat_cols, reg_num_cols = build_preprocessor(X_reg)

In [50]:
# Regression experiment function
def run_regression_experiment(run_name, eta0, max_iter, learning_rate):
    with mlflow.start_run(run_name=run_name):
        model = Pipeline([
            ("preprocessor", preprocessor_reg),
            ("regressor", SGDRegressor(
                loss="squared_error",
                eta0=eta0,
                max_iter=max_iter,
                learning_rate=learning_rate,
                tol=1e-3,
                random_state=42
            ))
        ])

        model.fit(X_train_reg, y_train_reg)

        y_train_pred = model.predict(X_train_reg)
        y_test_pred = model.predict(X_test_reg)

        train_mse = mean_squared_error(y_train_reg, y_train_pred)
        test_mse = mean_squared_error(y_test_reg, y_test_pred)
        train_mae = mean_absolute_error(y_train_reg, y_train_pred)
        test_mae = mean_absolute_error(y_test_reg, y_test_pred)
        train_r2 = r2_score(y_train_reg, y_train_pred)
        test_r2 = r2_score(y_test_reg, y_test_pred)

        cv = KFold(n_splits=5, shuffle=True, random_state=42)
        cv_mse = -cross_val_score(
            model, X_train_reg, y_train_reg,
            cv=cv, scoring="neg_mean_squared_error"
        ).mean()

        mlflow.log_param("task", "regression")
        mlflow.log_param("model_type", "SGDRegressor")
        mlflow.log_param("eta0", eta0)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("train_rows", len(X_train_reg))
        mlflow.log_param("test_rows", len(X_test_reg))
        mlflow.log_param("num_features_before_encoding", X_reg.shape[1])

        mlflow.log_metric("train_mse", train_mse)
        mlflow.log_metric("test_mse", test_mse)
        mlflow.log_metric("train_mae", train_mae)
        mlflow.log_metric("test_mae", test_mae)
        mlflow.log_metric("train_r2", train_r2)
        mlflow.log_metric("test_r2", test_r2)
        mlflow.log_metric("cv_mse", cv_mse)

        # save artifact
        artifact_path = f"../artifacts/{run_name}_regression_model.joblib"
        joblib.dump(model, artifact_path)
        mlflow.log_artifact(artifact_path)

        # log sklearn model directly to mlflow
        mlflow.sklearn.log_model(model, artifact_path="model")

        print(f"Completed: {run_name}")
        print("Test MSE:", test_mse, "| Test MAE:", test_mae, "| Test R2:", test_r2)

In [51]:
# Run regression experiments
regression_experiments = [
    {"run_name": "reg_eta0001_iter500",  "eta0": 0.001, "max_iter": 500,  "learning_rate": "constant"},
    {"run_name": "reg_eta0001_iter1000", "eta0": 0.001, "max_iter": 1000, "learning_rate": "constant"},
    {"run_name": "reg_eta001_iter1000",  "eta0": 0.01,  "max_iter": 1000, "learning_rate": "constant"},
    {"run_name": "reg_eta005_iter1000",  "eta0": 0.05,  "max_iter": 1000, "learning_rate": "constant"},
    {"run_name": "reg_optimal_iter1000", "eta0": 0.01,  "max_iter": 1000, "learning_rate": "optimal"},
]

for exp in regression_experiments:
    run_regression_experiment(**exp)

2026/04/05 13:35:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/05 13:35:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Completed: reg_eta0001_iter500
Test MSE: 20617.44830753182 | Test MAE: 123.6256361499294 | Test R2: 0.0004888252246697533


2026/04/05 13:35:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/05 13:35:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Completed: reg_eta0001_iter1000
Test MSE: 20617.44830753182 | Test MAE: 123.6256361499294 | Test R2: 0.0004888252246697533


2026/04/05 13:35:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/05 13:35:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Completed: reg_eta001_iter1000
Test MSE: 22088.57904874433 | Test MAE: 126.69312337382243 | Test R2: -0.07082997201272923


2026/04/05 13:35:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/05 13:35:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Completed: reg_eta005_iter1000
Test MSE: 31152.91980703652 | Test MAE: 144.37230443014397 | Test R2: -0.5102592236226307


2026/04/05 13:36:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/05 13:36:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Completed: reg_optimal_iter1000
Test MSE: 101414427653.52844 | Test MAE: 185042.5113700683 | Test R2: -4916458.700113135


                Part B — Delivery Status Classification Experiments

In [52]:
# Prepare delivery status data
status_df = df.copy()

y_status = status_df["delivery_status"]
X_status = status_df.drop(columns=["delivery_status", "customer_segment", "price"], errors="ignore")

X_train_status, X_test_status, y_train_status, y_test_status = train_test_split(
    X_status, y_status, test_size=0.2, random_state=42, stratify=y_status
)

preprocessor_status, status_cat_cols, status_num_cols = build_preprocessor(X_status)

print("Delivery status class distribution:")
print(y_status.value_counts())

Delivery status class distribution:
delivery_status
Delivered    7047
Shipped      1938
Pending       513
Returned      502
Name: count, dtype: int64


In [53]:
# Delivery status experiment function
def run_status_experiment(run_name, max_iter, class_weight):
    with mlflow.start_run(run_name=run_name):
        model = Pipeline([
            ("preprocessor", preprocessor_status),
            ("classifier", LogisticRegression(
                solver="lbfgs",
                max_iter=max_iter,
                class_weight=class_weight,
                random_state=42
            ))
        ])

        model.fit(X_train_status, y_train_status)

        y_pred = model.predict(X_test_status)

        accuracy = accuracy_score(y_test_status, y_pred)
        f1_macro = f1_score(y_test_status, y_pred, average="macro")
        precision_macro = precision_score(y_test_status, y_pred, average="macro", zero_division=0)
        recall_macro = recall_score(y_test_status, y_pred, average="macro", zero_division=0)

        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_f1_macro = cross_val_score(
            model, X_train_status, y_train_status,
            cv=cv, scoring="f1_macro"
        ).mean()

        report = classification_report(y_test_status, y_pred, output_dict=True, zero_division=0)

        mlflow.log_param("task", "classification_delivery_status")
        mlflow.log_param("model_type", "LogisticRegression")
        mlflow.log_param("solver", "lbfgs")
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("class_weight", str(class_weight))

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("f1_macro", f1_macro)
        mlflow.log_metric("precision_macro", precision_macro)
        mlflow.log_metric("recall_macro", recall_macro)
        mlflow.log_metric("cv_f1_macro", cv_f1_macro)

        # log per-class recall if available
        for cls in report:
            if isinstance(report[cls], dict) and "recall" in report[cls]:
                safe_name = str(cls).replace(" ", "_")
                mlflow.log_metric(f"recall_{safe_name}", report[cls]["recall"])

        artifact_path = f"../artifacts/{run_name}_delivery_status_model.joblib"
        joblib.dump(model, artifact_path)
        mlflow.log_artifact(artifact_path)
        mlflow.sklearn.log_model(model, artifact_path="model")

        print(f"Completed: {run_name}")
        print("Accuracy:", accuracy, "| F1 Macro:", f1_macro)

In [54]:
# Run delivery status experiments
status_experiments = [
    {"run_name": "status_balanced_iter500",  "max_iter": 500,  "class_weight": "balanced"},
    {"run_name": "status_balanced_iter1000", "max_iter": 1000, "class_weight": "balanced"},
    {"run_name": "status_none_iter1000",     "max_iter": 1000, "class_weight": None},
]

for exp in status_experiments:
    run_status_experiment(**exp)

2026/04/05 13:36:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/05 13:36:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Completed: status_balanced_iter500
Accuracy: 0.204 | F1 Macro: 0.16972046893668824


2026/04/05 13:36:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/05 13:36:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Completed: status_balanced_iter1000
Accuracy: 0.204 | F1 Macro: 0.16972046893668824


2026/04/05 13:36:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/05 13:36:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Completed: status_none_iter1000
Accuracy: 0.7045 | F1 Macro: 0.2066588442358463


                Part C — Customer Segment Classification Experiments

In [55]:
# Prepare customer segment data
segment_df = df.copy()

y_segment = segment_df["customer_segment"]
X_segment = segment_df.drop(columns=["customer_segment", "delivery_status", "price"], errors="ignore")

X_train_segment, X_test_segment, y_train_segment, y_test_segment = train_test_split(
    X_segment, y_segment, test_size=0.2, random_state=42, stratify=y_segment
)

preprocessor_segment, segment_cat_cols, segment_num_cols = build_preprocessor(X_segment)

print("Customer segment class distribution:")
print(y_segment.value_counts())

Customer segment class distribution:
customer_segment
VIP          5149
Returning    4285
New           566
Name: count, dtype: int64


In [56]:
# Customer segment experiment function
def run_segment_experiment(run_name, max_iter, class_weight, solver="lbfgs"):
    with mlflow.start_run(run_name=run_name):
        model = Pipeline([
            ("preprocessor", preprocessor_segment),
            ("classifier", LogisticRegression(
                solver=solver,
                max_iter=max_iter,
                class_weight=class_weight,
                random_state=42
            ))
        ])

        model.fit(X_train_segment, y_train_segment)

        y_pred = model.predict(X_test_segment)
        y_prob = model.predict_proba(X_test_segment)

        accuracy = accuracy_score(y_test_segment, y_pred)
        f1_macro = f1_score(y_test_segment, y_pred, average="macro")
        precision_macro = precision_score(y_test_segment, y_pred, average="macro", zero_division=0)
        recall_macro = recall_score(y_test_segment, y_pred, average="macro", zero_division=0)

        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_f1_macro = cross_val_score(
            model, X_train_segment, y_train_segment,
            cv=cv, scoring="f1_macro"
        ).mean()

        report = classification_report(y_test_segment, y_pred, output_dict=True, zero_division=0)

        mlflow.log_param("task", "classification_customer_segment")
        mlflow.log_param("model_type", "LogisticRegression")
        mlflow.log_param("solver", solver)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("class_weight", str(class_weight))

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("f1_macro", f1_macro)
        mlflow.log_metric("precision_macro", precision_macro)
        mlflow.log_metric("recall_macro", recall_macro)
        mlflow.log_metric("cv_f1_macro", cv_f1_macro)

        for cls in report:
            if isinstance(report[cls], dict) and "precision" in report[cls]:
                safe_name = str(cls).replace(" ", "_")
                mlflow.log_metric(f"precision_{safe_name}", report[cls]["precision"])
                mlflow.log_metric(f"recall_{safe_name}", report[cls]["recall"])
                mlflow.log_metric(f"f1_{safe_name}", report[cls]["f1-score"])

        artifact_path = f"../artifacts/{run_name}_customer_segment_model.joblib"
        joblib.dump(model, artifact_path)
        mlflow.log_artifact(artifact_path)
        mlflow.sklearn.log_model(model, artifact_path="model")

        print(f"Completed: {run_name}")
        print("Accuracy:", accuracy, "| F1 Macro:", f1_macro)

In [57]:
# Run customer segment experiments
segment_experiments = [
    {"run_name": "segment_balanced_iter500",  "max_iter": 500,  "class_weight": "balanced"},
    {"run_name": "segment_balanced_iter1000", "max_iter": 1000, "class_weight": "balanced"},
    {"run_name": "segment_none_iter1000",     "max_iter": 1000, "class_weight": None},
]

for exp in segment_experiments:
    run_segment_experiment(**exp)

2026/04/05 13:36:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/05 13:36:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Completed: segment_balanced_iter500
Accuracy: 0.3005 | F1 Macro: 0.2696908395040453


2026/04/05 13:37:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/05 13:37:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Completed: segment_balanced_iter1000
Accuracy: 0.3005 | F1 Macro: 0.2696908395040453


2026/04/05 13:37:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/05 13:37:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Completed: segment_none_iter1000
Accuracy: 0.509 | F1 Macro: 0.24633136158662527


                Part D — Pick the Best Final Customer Segment Model

In [58]:
# Train and save final selected model
best_segment_model = Pipeline([
    ("preprocessor", preprocessor_segment),
    ("classifier", LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

best_segment_model.fit(X_train_segment, y_train_segment)

final_model_path = "../artifacts/model.joblib"
joblib.dump(best_segment_model, final_model_path)

print("Final deployment model saved at:", final_model_path)

Final deployment model saved at: ../artifacts/model.joblib


In [59]:
# Log final chosen model separately
with mlflow.start_run(run_name="final_selected_customer_segment_model"):
    y_pred_final = best_segment_model.predict(X_test_segment)

    accuracy_final = accuracy_score(y_test_segment, y_pred_final)
    f1_macro_final = f1_score(y_test_segment, y_pred_final, average="macro")
    precision_macro_final = precision_score(y_test_segment, y_pred_final, average="macro", zero_division=0)
    recall_macro_final = recall_score(y_test_segment, y_pred_final, average="macro", zero_division=0)

    mlflow.log_param("task", "final_customer_segment_model")
    mlflow.log_param("solver", "lbfgs")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("class_weight", "balanced")

    mlflow.log_metric("accuracy", accuracy_final)
    mlflow.log_metric("f1_macro", f1_macro_final)
    mlflow.log_metric("precision_macro", precision_macro_final)
    mlflow.log_metric("recall_macro", recall_macro_final)

    mlflow.log_artifact(final_model_path)
    mlflow.sklearn.log_model(best_segment_model, artifact_path="model")

    print("Final model logged successfully.")

2026/04/05 13:37:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/05 13:37:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Final model logged successfully.


                Part E — Compare Runs in MLflow UI

## How to open the MLflow UI

We run this command in our terminal from the project folder:

```bash
mlflow ui --backend-store-uri ../mlruns

In [60]:
with mlflow.start_run():
    ...

import os
import mlflow
import mlflow.sklearn

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Make notebook log to the same place your UI should read from
mlflow.set_tracking_uri("file:" + os.path.abspath("../mlruns"))
mlflow.set_experiment("AuraCart_ML_Experiments")

# Make sure these variables already exist from earlier cells:
# X_train_segment, X_test_segment, y_train_segment, y_test_segment

categorical_cols = X_train_segment.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_cols = X_train_segment.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])

with mlflow.start_run(run_name="customer_segment_test_run"):
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            solver="lbfgs",
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    model.fit(X_train_segment, y_train_segment)
    y_pred = model.predict(X_test_segment)
    acc = accuracy_score(y_test_segment, y_pred)

    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("solver", "lbfgs")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metric("accuracy", acc)

    mlflow.sklearn.log_model(model, "model")

    print("Run completed. Accuracy:", acc)

2026/04/05 13:37:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/05 13:37:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run completed. Accuracy: 0.3005


## Task 4.2 + 4.3

In this notebook, we prepare the final machine learning model for deployment.

We will:
- build a pipeline (preprocessing + model)
- train the model
- save it as model.joblib
- create requirements.txt
- test the model

In [61]:
print("Task 4.2 + 4.3 - Model Packaging and Deployment")

Task 4.2 + 4.3 - Model Packaging and Deployment


In [74]:
# ==============================
# Task 4.2 + 4.3 - Deployment
# ==============================

import os
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# ------------------------------
# Step 1: Create artifacts folder
# ------------------------------
os.makedirs("../artifacts", exist_ok=True)

# ------------------------------
# Step 2: Load dataset
# ------------------------------
df = pd.read_csv("../data/cleaned_data.csv")

df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print("Dataset loaded:", df.shape)

# ------------------------------
# Step 3: Prepare data
# ------------------------------
# TARGET = customer_segment (VERY IMPORTANT)
y = df["customer_segment"]

# Drop unnecessary columns
X = df.drop(columns=["customer_segment", "delivery_status", "price"], errors="ignore")

# ------------------------------
# Step 4: Train/Test split
# ------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ------------------------------
# Step 5: Preprocessing
# ------------------------------
categorical_cols = X.select_dtypes(include=["object"]).columns
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])

# ------------------------------
# Step 6: Final model (Pipeline)
# ------------------------------
final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# ------------------------------
# Step 7: Train model
# ------------------------------
final_model.fit(X_train, y_train)

print("Model trained successfully")

# ------------------------------
# Step 8: Save model
# ------------------------------
joblib.dump(final_model, "../artifacts/model.joblib")

print("Model saved as model.joblib")

# ------------------------------
# Step 9: Create requirements.txt
# ------------------------------
with open("../artifacts/requirements.txt", "w") as f:
    f.write("pandas\nnumpy\nscikit-learn\njoblib\n")

print("requirements.txt created")

# ------------------------------
# Step 10: Test model
# ------------------------------
loaded_model = joblib.load("../artifacts/model.joblib")

sample = X_test.iloc[:1]

prediction = loaded_model.predict(sample)

print("Sample prediction:", prediction)

Dataset loaded: (10000, 17)
Model trained successfully
Model saved as model.joblib
requirements.txt created
Sample prediction: ['VIP']
